In [11]:
# PyTorch functions/methods helpers

# 8.2.1
def vgg_block(in_channels, out_channels, num_convs):

    layers = []
    current_channels = in_channels

    for _ in range(num_convs): # For each layer (layer count = num_convs)
        layers.append(nn.Conv2d(current_channels, out_channels, kernel_size=3, padding=1)) # Perform a Conv2d layer first, then perform a ReLU right after
        layers.append(nn.ReLU())
        current_channels = out_channels # Tie in the current_channels to out_channels argument so that future layers have the right in_channels input (while out_channels is specified by user argument)

    layers.append(nn.MaxPool2d(kernel_size=2, stride=2)) # After all convolutional layers, halve the spatial dimensions using 2×2 max and return the maximum elements within each pool
    return nn.Sequential(*layers) # * unpacks the list into individual arguments

* VGG turns CNN architecture into a block design discipline.
* Instead of treating every layer as a one-off decision, it **repeatedly stacks small 3 by 3 convolutions followed by pooling**.
* That makes the architecture easier to read, modify, and scale.

# How to use this notebook

* Run the notebook from top to bottom in a clean kernel.

* The code uses small synthetic tensors so that architecture mechanics can be inspected without downloads, `torchvision`, ImageNet-scale images, or long training runs.

* Before important cells, predict the shape, parameter count, or failure mode, then read the assertions as executable contracts.

# You are done when you can

- define a VGG block and explain why it is reusable
- compare stacked 3 by 3 convolutions with a larger single convolution
- build a small VGG-style classifier from an architecture configuration
- trace spatial downsampling through repeated blocks
- debug the effect of forgetting padding in a VGG block

In [3]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

def trace_module_shapes(module, X):
    rows = []
    current = X
    for name, layer in module.named_children():
        current = layer(current)
        rows.append((name, layer.__class__.__name__, shape(current)))
    return rows, current

# 8.2.0 The Problem This Notebook Solves

VGG's central lesson is not only a particular network. It is a way of designing networks:

```text
choose a simple block
repeat the block
increase channels across stages
reduce spatial resolution between stages
```

A block is a reusable module pattern. Chapter 6 introduced modules as software objects.

VGG shows why this matters architecturally: **repeated blocks let you express a deep model with a small, inspectable design vocabulary**.

# Why does VGG increase channels while reducing spatial resolution?

Channels represent **what features are present**, while spatial dimensions represent **where those features are**.

VGG progressively trades spatial precision for richer feature representations:

`224×224×64 → 112×112×128 → 56×56×256 → 28×28×512`

As the network gets deeper, it cares less about exact pixel locations and more about higher-level structures.

Increasing channels gives it more capacity to represent these features, while reducing spatial resolution:

- Reduces compute and memory
- Increases the effective receptive field
- Sacrifices fine spatial detail that classification generally doesn't need

For example, increasing `28×28×512` to `56×56×512` would require 4× as many activations. Higher spatial resolution is useful for tasks like segmentation and detection, but classification mainly needs to know **what is present**, not its exact pixel location.

> **VGG trades some "where" information for richer "what" representations as the network gets deeper.**

# 8.2.1 A VGG Block Is Repeated Local Processing Plus Downsampling

A typical VGG block contains:

- one or more 3 by 3 convolution layers with padding 1
- ReLU after each convolution
- 2 by 2 max pooling with stride 2

Padding 1 is important.
* A 3 by 3 convolution with padding 1 preserves height and width before pooling.
* Then pooling halves the spatial size.
* This makes the block's shape contract predictable.

Before running the cell, predict:

- Input shape: `(2, 3, 32, 32)`.
- Two padded convolutions should keep `32 by 32`.
- Pooling should produce `16 by 16`.
- Output channel count should be 8.

In [4]:
def vgg_block(in_channels, out_channels, num_convs):

    layers = []
    current_channels = in_channels

    for _ in range(num_convs): # For each layer (layer count = num_convs)
        layers.append(nn.Conv2d(current_channels, out_channels, kernel_size=3, padding=1)) # Perform a Conv2d layer first, then perform a ReLU right after
        layers.append(nn.ReLU())
        current_channels = out_channels # Tie in the current_channels to out_channels argument so that future layers have the right in_channels input (while out_channels is specified by user argument)

    layers.append(nn.MaxPool2d(kernel_size=2, stride=2)) # After all convolutional layers, halve the spatial dimensions using 2×2 max and return the maximum elements within each pool
    return nn.Sequential(*layers) # * unpacks the list into individual arguments

block = vgg_block(3, 8, num_convs=2)
X = torch.randn(2, 3, 32, 32)
Y = block(X)

print("output shape:", shape(Y))
assert shape(Y) == (2, 8, 16, 16) # batch_size remains 2; out_channels updated from 3 → 8 (user defined vgg_block()), spatial dimensions are halved from 32×32 → 16×16 by MaxPool2d

output shape: (2, 8, 16, 16)


# 8.2.2 Two 3 by 3 Convolutions See a 5 by 5 Neighborhood

Stacking small kernels increases the effective receptive field.
* Receptive field means the region of the original input that can affect one output value.
* 2 stride-1, 3 by 3 convolutions let a later output depend on a 5 by 5 region, but with an extra nonlinearity between the 2 convolutions.

For equal input and output channel width, two 3 by 3 convolutions often use fewer parameters than one 5 by 5 convolution:

```text
two 3 by 3 layers: 2 * 3 * 3 * C * C
one 5 by 5 layer: 5 * 5 * C * C
```

This is a design tradeoff: VGG prefers a regular stack of small local operations.

## Why does VGG stack 3×3 convolutions?

A **receptive field** is the region of the original input that can affect one output value.

For stride-1 convolutions:

`R_new = R_old + (kernel_size - 1)`

Therefore:

`3×3 → 3×3` gives `1 → 3 → 5`, so two 3×3 convolutions give a later output access to a **5×5 region** of the original image. A third 3×3 would expand it to `7×7`.

VGG prefers this over one large convolution because:

- Two 3×3 convolutions use fewer parameters than one 5×5: `18C²` vs. `25C²`
- Two convolutions provide an **extra ReLU/nonlinearity**, allowing the network to learn a more complex transformation
- It keeps the architecture simple and repetitive

```text
3×3 Conv → ReLU → 3×3 Conv → ReLU
        ↓
effective receptive field = 5×5

In [5]:
channels = 16
two_3x3 = 2 * 3 * 3 * channels * channels
one_5x5 = 5 * 5 * channels * channels

print("two 3x3 weights:", two_3x3)
print("one 5x5 weights:", one_5x5)
print("saving:", one_5x5 - two_3x3)

assert two_3x3 < one_5x5

two 3x3 weights: 4608
one 5x5 weights: 6400
saving: 1792


# 8.2.3 Build VGG From an Architecture Configuration

An architecture configuration is a compact description of repeated stages. In this notebook, each tuple means:

```text
(number of convolutions in the block, output channels)
```

The builder below converts that list into an executable `nn.Sequential`. This is the bridge from design vocabulary to framework code.

Before running the cell, predict:

- Three blocks with pooling should reduce 64 to 32 to 16 to 8.
- The final adaptive average pool should make the dense head independent of the exact final spatial size.
- The logits should have shape `(2, 10)`.

```text
AlexNet:
  learn increasingly useful representations
  ↓
  reduce spatial resolution

VGG:
  same general progression
  +
  use repeated small 3×3 conv blocks
  +
  increase channels systematically

This tutorial's make_vgg():
  ↑
  Python convenience for automatically assembling those blocks
```



In [6]:
def make_vgg(in_channels, arch, num_classes=10):

    layers = []
    current_channels = in_channels

    for num_convs, out_channels in arch:
        layers.append(vgg_block(current_channels, out_channels, num_convs))
        current_channels = out_channels

    # vgg_block(1, 8, 1)
    #     ↓
    # Conv 1→8
    # ReLU
    # Pool

    # vgg_block(8, 16, 1)
    #         ↓
    # Conv 8→16
    # ReLU
    # Pool

    # vgg_block(16, 32, 2)
    #         ↓
    # Conv 16→32
    # ReLU
    # Conv 32→32
    # ReLU
    # Pool

    layers += [
        nn.AdaptiveAvgPool2d((1, 1)), # Reduces each channel's spatial dimensions (H×W) to 1×1 by averaging across its spatial elements
        nn.Flatten(),                 # Flattens (batch, channels, 1, 1) into (batch, channels)
        nn.Linear(current_channels, num_classes),
    ]
    return nn.Sequential(*layers)

    # (2, 32, 8, 8)
    #    ↓ AdaptiveAvgPool
    # (2, 32, 1, 1)    ← average each channel spatially
    #       ↓ Flatten
    # (2, 32)          ← turn each channel into one feature
    #       ↓ Linear
    # (2, 10)          ← 10 class scores

tiny_vgg = make_vgg(1, [(1, 8), (1, 16), (2, 32)])
X = torch.randn(2, 1, 64, 64)
rows, logits = trace_module_shapes(tiny_vgg, X)
for row in rows:
    print(row)

assert shape(logits) == (2, 10)

('0', 'Sequential', (2, 8, 32, 32))
('1', 'Sequential', (2, 16, 16, 16))
('2', 'Sequential', (2, 32, 8, 8))
('3', 'AdaptiveAvgPool2d', (2, 32, 1, 1))
('4', 'Flatten', (2, 32))
('5', 'Linear', (2, 10))


# 8.2.4 Blocks Make Parameter Accounting Local

When a network is built from blocks, you can inspect each block separately.

This matters because modern CNNs are not just long lists of layers; they are systems of repeated components.

The cell counts trainable scalars per top-level stage.

This is not the same as measuring runtime speed, but it reveals where model capacity lives.

In [9]:
for name, layer in tiny_vgg.named_children():
    print(name, layer.__class__.__name__, count_parameters(layer))

total = count_parameters(tiny_vgg)
print("total trainable scalars:", total)

assert total == sum(count_parameters(layer) for layer in tiny_vgg.children())

0 Sequential 80
1 Sequential 1168
2 Sequential 13888
3 AdaptiveAvgPool2d 0
4 Flatten 0
5 Linear 330
total trainable scalars: 15466


# 8.2.5 Break It Deliberately: Forget Padding

The VGG block pattern depends on padded 3 by 3 convolutions preserving spatial size before pooling.

**If you forget padding, every convolution shrinks height and width before the pooling layer runs**.

The forward pass may still run, which makes this bug subtle. The architecture no longer has the shape story you intended.

In [10]:
bad_block = nn.Sequential(
    nn.Conv2d(3, 8, kernel_size=3), nn.ReLU(),
    nn.Conv2d(8, 8, kernel_size=3), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
)

good_Y = block(torch.randn(2, 3, 32, 32))
bad_Y = bad_block(torch.randn(2, 3, 32, 32))

print("good block:", shape(good_Y)) # The original vgg_block had Conv2d w/ padding + ReLU, then finalize with MaxPool2d, so the final shape contract is exactly halved
print("bad block:", shape(bad_Y)) # The new block omitted padding at Conv2d, causing the final shape contract to be halved - 2*1 (edges on both sides)

try:
    assert shape(bad_Y)[-2:] == (16, 16)
except AssertionError:
    print("The block ran, but it violated the intended VGG shape contract.")

good block: (2, 8, 16, 16)
bad block: (2, 8, 14, 14)
The block ran, but it violated the intended VGG shape contract.


# 8.2 Checkpoint

Answer these before moving on.

Short markdown answers in the notebook are enough; the checkpoint is meant to test whether you can explain the mechanics without rereading the code.

1. What is a VGG block?
> A VGG block is a reusable pattern of one or more 3×3 convolutions with ReLU, followed by 2×2 max pooling. The convolutions learn features while preserving spatial size, and the pooling reduces the spatial dimensions.

2. Why can repeated 3 by 3 convolutions be preferable to one larger convolution?
> Repeated 3×3 convolutions can achieve a larger receptive field with fewer parameters than one larger convolution. They also add an extra ReLU nonlinearity between the convolutions, allowing the network to learn more complex features

3. Why does padding matter inside a VGG block?
> Padding 1 allows a 3×3 convolution to preserve the height and width of the feature map. This makes the VGG block's shape predictable: the convolutions preserve the spatial dimensions, and the 2×2 pooling layer halves them

4. What does an architecture configuration buy you as a programmer?
> An architecture configuration lets me describe the network compactly and generate it programmatically. I can change the number of convolutions or channels without manually rewriting every layer, making the model easier to modify and scale

5. Why is a running forward pass not enough to prove the architecture is correct?
> A forward pass only proves that the operations can execute on the given input. **It does not prove that the architecture has the intended shapes, parameter counts, or design behavior**. Assertions and shape/parameter checks can verify those contracts